[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/huggingface-nlp-certified/notebooks/day-01-hf-ecosystem.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · The Hugging Face Ecosystem — Hub, Libraries, and First Pipeline
**certified-journeys / huggingface-nlp-certified** · Day 1 · Getting Started

> **Goal for today:** Run your first Hugging Face pipeline, navigate the Hub to find models and datasets, and read a model card critically so you know what a model can and cannot do.


## Step 1 · Install the Core Libraries

The Hugging Face ecosystem is built on four main packages:

| Package | Role |
|---|---|
| `transformers` | Models, tokenizers, pipelines |
| `datasets` | Fast dataset loading and processing |
| `evaluate` | Standardised metrics (accuracy, F1, BLEU …) |
| `accelerate` | Distributed training and mixed precision |

All four are maintained by Hugging Face and follow a consistent API design. You will use `transformers` the most on Day 1, but the others appear frequently throughout the course.


In [ ]:
%pip install -q transformers datasets evaluate accelerate


## Step 2 · The Hugging Face Hub at a Glance

The Hub (huggingface.co) is more than a model zoo. It hosts:

- **Models** — 400k+ pre-trained checkpoints across tasks and languages
- **Datasets** — 50k+ datasets with automatic versioning and streaming
- **Spaces** — Live demo apps (Gradio or Streamlit) anyone can fork
- **Model cards** — Structured documentation: intended use, training data, limitations, evaluation results

The `huggingface_hub` library (installed as a dependency of `transformers`) lets you interact with the Hub programmatically — search, filter, and download without leaving Python.


In [ ]:
from huggingface_hub import list_models

# List the top-5 most downloaded English text-classification models
models = list(
    list_models(
        task="text-classification",
        language="en",
        sort="downloads",
        direction=-1,
        limit=5,
    )
)

print(f"{'Model ID':<50} {'Downloads':>12}")
print("-" * 65)
for m in models:
    # downloads may be None for very new models
    dl = getattr(m, "downloads", None) or 0
    print(f"{m.modelId:<50} {dl:>12,}")


### What just happened?

- `list_models` queries the Hub REST API and returns `ModelInfo` objects — no weights are downloaded.
- **`sort="downloads"` with `direction=-1`** gives descending order (most popular first).
- The Hub has search filters for task, language, library, dataset, license, and more — you can chain them.
- **Key insight:** popularity ≠ best fit. A model with 10 M downloads may be outdated; always check the model card date and evaluation scores.


## Step 3 · Your First Pipeline in Three Lines

The `pipeline` abstraction wraps tokenisation, model inference, and post-processing into a single callable. It is the fastest way to go from text to predictions without writing any boilerplate.

**Under the hood**, `pipeline('sentiment-analysis')` does:
1. Downloads the default checkpoint for that task (cached in `~/.cache/huggingface/`).
2. Loads the associated tokenizer.
3. Runs inference and maps logits to human-readable labels + scores.

The default sentiment model is `distilbert-base-uncased-finetuned-sst-2-english` — small, fast, and accurate enough for prototyping.


In [ ]:
from transformers import pipeline

# The classic three-liner
classifier = pipeline("sentiment-analysis")
result = classifier("I love NLP!")
print(result)

# Pipelines also handle batches — pass a list
batch_results = classifier([
    "Hugging Face makes NLP accessible to everyone.",
    "This model card is missing critical information about bias.",
    "The training took three weeks on 64 A100s.",
])

print("\nBatch results:")
for text, res in zip(
    ["Accessible NLP", "Missing bias info", "Long training"],
    batch_results
):
    print(f"  {text:<20} → {res['label']:<10} (score: {res['score']:.3f})")


### What just happened?

- The pipeline returned `[{'label': 'POSITIVE', 'score': 0.9998}]` — a label and a confidence score.
- **Batching is free**: passing a list runs the tokenizer and model in a single forward pass, which is significantly faster than calling the pipeline in a loop.
- The score is a softmax probability, not raw logit — it always sums to 1.0 across all classes.
- **Key insight:** For production, always specify the model explicitly (`pipeline('sentiment-analysis', model='...')`) — the default checkpoint can change across `transformers` versions, breaking reproducibility.


## Step 4 · Specifying a Model Explicitly

Instead of relying on the default checkpoint, pass any Hub model ID. This is best practice for reproducibility: the same model ID always resolves to the same weights (unless the author force-pushes, which is rare).

Below we use `cardiffnlp/twitter-roberta-base-sentiment-latest` — a RoBERTa model fine-tuned on tweets, with three-way sentiment (NEGATIVE / NEUTRAL / POSITIVE).


In [ ]:
from transformers import pipeline

# Pin a specific model for reproducibility
twitter_clf = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    # top_k=None returns scores for ALL labels, not just the winner
    top_k=None,
)

texts = [
    "Just shipped a new feature — feeling great! 🚀",
    "The deployment failed again. Third time this week.",
    "Pushed the PR. Waiting for CI to finish.",
]

for text in texts:
    scores = twitter_clf(text)[0]  # list of {label, score} dicts
    # sort by score descending
    scores_sorted = sorted(scores, key=lambda x: x["score"], reverse=True)
    top = scores_sorted[0]
    print(f"Text : {text[:55]}")
    print(f"  Top: {top['label']} ({top['score']:.3f})")
    print(f"  All: { {s['label']: round(s['score'], 3) for s in scores_sorted} }")
    print()


### What just happened?

- `top_k=None` returns **all label scores**, not just the argmax — useful when you need calibrated probabilities, not just a hard prediction.
- **Neutral class matters**: many real-world statements ("Waiting for CI") are genuinely neutral; a binary model would force one polarity.
- RoBERTa vs BERT: RoBERTa drops next-sentence prediction and uses dynamic masking during training — generally better on short informal text like tweets.
- **Key insight:** Always match your model to your domain. A model trained on tweets behaves differently on formal news text.


## Step 5 · Reading a Model Card Programmatically

A model card is the primary documentation artifact on the Hub. For `bert-base-uncased`, the card covers:

- **Model description** — architecture, parameter count, pre-training objective
- **Training data** — BookCorpus + English Wikipedia (~16 GB)
- **Intended use** — feature extraction and fine-tuning; NOT meant for direct generation
- **Limitations** — English-only; reflects biases in Wikipedia and BookCorpus (gender, race, etc.)
- **Evaluation** — GLUE benchmark scores

The `huggingface_hub` library can fetch the card content so you can audit it without leaving Python.


In [ ]:
from huggingface_hub import ModelCard, model_info

model_id = "bert-base-uncased"

# Fetch structured metadata (architecture, tags, downloads, likes)
info = model_info(model_id)
print("=== Model Metadata ===")
print(f"  Model ID      : {info.modelId}")
print(f"  Pipeline tag  : {info.pipeline_tag}")
print(f"  Library       : {info.library_name}")
print(f"  Downloads/mo  : {getattr(info, 'downloads', 'N/A')}")
print(f"  Likes         : {getattr(info, 'likes', 'N/A')}")
print(f"  Tags          : {info.tags[:8]}")

# Fetch the full model card text
card = ModelCard.load(model_id)

# Print the first 60 lines (the summary + intended use)
lines = card.content.split("\n")
print("\n=== Model Card (first 60 lines) ===")
for line in lines[:60]:
    print(line)


### What just happened?

- `model_info()` returns structured metadata (tags, pipeline_tag, card_data) without downloading weights.
- `ModelCard.load()` fetches the raw Markdown text of the card — you can parse it, search it, or log it for auditing.
- **BERT's limitations section** explicitly notes biases from Wikipedia; this is why reading model cards is a professional responsibility, not just good practice.
- **Key insight:** Before using any model in a product, check the model card for: (1) training data provenance, (2) evaluation datasets and metrics, (3) known failure modes and demographic biases.


## Step 6 · Exploring the Dataset Hub

The `datasets` library can load any Hub dataset in one line. Key features:

- **Arrow-backed**: datasets are memory-mapped; you can work with 100 GB datasets without loading them into RAM.
- **Streaming**: `load_dataset(..., streaming=True)` iterates without downloading the full dataset.
- **Consistent schema**: every dataset exposes `train`/`validation`/`test` splits and a `features` dict.

Below we load the `emotion` dataset — 6-class English tweet emotion classification (joy, anger, fear, sadness, surprise, love).


In [ ]:
from datasets import load_dataset

# Load the emotion dataset (small, ~416k examples, downloads quickly)
emotion = load_dataset("dair-ai/emotion", split="train")

print("=== Dataset Overview ===")
print(f"  Type        : {type(emotion)}")
print(f"  Rows        : {len(emotion):,}")
print(f"  Features    : {emotion.features}")
print(f"  Column names: {emotion.column_names}")

print("\n=== First 5 Examples ===")
for i in range(5):
    row = emotion[i]
    label_name = emotion.features["label"].int2str(row["label"])
    print(f"  [{label_name:<9}] {row['text'][:70]}")

print("\n=== Label Distribution ===")
from collections import Counter
label_counts = Counter(emotion["label"])
for label_id, count in sorted(label_counts.items()):
    name = emotion.features["label"].int2str(label_id)
    bar = "█" * (count // 500)
    print(f"  {name:<10} {count:>6,}  {bar}")


### What just happened?

- `load_dataset` downloaded and cached the dataset as Apache Arrow files — subsequent runs load from cache instantly.
- `emotion.features` tells you the schema: `text` (string) and `label` (ClassLabel with 6 classes).
- **ClassLabel** has `.int2str()` and `.str2int()` — no manual mapping needed.
- **Key insight:** Always check the label distribution before fine-tuning. The emotion dataset is class-imbalanced (joy and sadness dominate); naive training may ignore rare classes like surprise.


## Challenge

Search the Hub for question-answering models trained on the `squad` dataset, filter to English-only, and print the top-3 by downloads. Then load one of them as a pipeline and answer a question from a short passage you provide.


In [ ]:
# Challenge: Find top QA models and run one
# Your solution here

from huggingface_hub import list_models
from transformers import pipeline

# Step 1: Search for QA models — hint: task="question-answering", language="en"
# qa_models = list(list_models(...))
# for m in qa_models[:3]:
#     print(m.modelId, getattr(m, 'downloads', 0))

# Step 2: Load a QA pipeline with one of the results
# qa = pipeline("question-answering", model="...")

# Step 3: Provide a context and a question
# context = "..."
# question = "..."
# result = qa(question=question, context=context)
# print(result)


---
## Day 1 Key Concepts Recap

| Concept | What to remember |
|---|---|
| Hub | Hosts models, datasets, Spaces, and model cards — not just weights |
| `pipeline()` | Fastest path from text to predictions; wraps tokenizer + model + post-processing |
| Default checkpoints | Convenient but can change across versions — always pin a model ID in production |
| `top_k=None` | Returns all label scores, not just the argmax |
| Model card | Read training data, limitations, and evaluation before deploying any model |
| `load_dataset` | Arrow-backed, cached, streaming-capable; check label distribution before training |

> **Tip:** The Hub is more than a model zoo — it hosts datasets, Spaces (demo apps), and model cards. Spend 10 minutes browsing before writing any code.

---
## What's next
**Day 2** → Tokenizers Deep Dive — BPE, WordPiece, and SentencePiece. You'll understand exactly how text becomes numbers before it reaches the model.

Mark Day 1 complete in your [tracker](../index.html).
